In [ ]:
# IMPORTANDO DEPENDÊNCIAS

from pymongo import MongoClient
import pandas as pd
from azure.storage.filedatalake import DataLakeServiceClient

from dotenv import load_dotenv
import os

In [5]:
# CRIANDO CONEXÃO COM O BANCO E PESQUISANDO COLEÇÃO

client = MongoClient("mongodb://localhost:27017/")
db = client["dados_PI4"]

print(db.list_collection_names())

['tendenciaEventos']


In [6]:
# BUSCANDO DADOS PARA VERIFICAR SE A CONEXÃO FUNCIONOU

colecao = db["tendenciaEventos"]

dados = colecao.find().limit(5)

for doc in dados:
    print(doc)

{'_id': ObjectId('6a0f46e513c507506239b83f'), 'data': datetime.datetime(2006, 1, 1, 0, 0), 'tipo_evento': 'casamento', 'valor': 56}
{'_id': ObjectId('6a0f46e513c507506239b840'), 'data': datetime.datetime(2006, 2, 1, 0, 0), 'tipo_evento': 'casamento', 'valor': 43}
{'_id': ObjectId('6a0f46e513c507506239b841'), 'data': datetime.datetime(2006, 3, 1, 0, 0), 'tipo_evento': 'casamento', 'valor': 46}
{'_id': ObjectId('6a0f46e513c507506239b842'), 'data': datetime.datetime(2006, 4, 1, 0, 0), 'tipo_evento': 'casamento', 'valor': 41}
{'_id': ObjectId('6a0f46e513c507506239b843'), 'data': datetime.datetime(2006, 5, 1, 0, 0), 'tipo_evento': 'casamento', 'valor': 46}


In [7]:
# CRIANDO O DATAFRAME

dados = list(colecao.find())
df = pd.DataFrame(dados)

# Converter _id
df["_id"] = df["_id"].astype(str)

df

,_id,data,tipo_evento,valor
0,6a0f46e513c507506239b83f,2006-01-01,casamento,56
1,6a0f46e513c507506239b840,2006-02-01,casamento,43
2,6a0f46e513c507506239b841,2006-03-01,casamento,46
3,6a0f46e513c507506239b842,2006-04-01,casamento,41
4,6a0f46e513c507506239b843,2006-05-01,casamento,46
...,...,...,...,...
1710,6a0f46e513c507506239beed,2026-01-01,formatura,27
1711,6a0f46e513c507506239beee,2026-02-01,formatura,26
1712,6a0f46e513c507506239beef,2026-03-01,formatura,27
1713,6a0f46e513c507506239bef0,2026-04-01,formatura,18


In [8]:
# CONVERTENDO DF PARA PARQUET

df.to_parquet("tendencia_eventos.parquet", index=False)

In [9]:
# CRIANDO VARIÁVEIS DE AMBIENTE E VERIFICANDO SE O DF FOI TRANSFORMADO EM PARQUET

load_dotenv()

account_name = os.getenv("AZURE_STORAGE_ACCOUNT")
account_key = os.getenv("AZURE_STORAGE_KEY")

file_system = os.getenv("AZURE_FILE_SYSTEM")
directory = os.getenv("AZURE_DIRECTORY")
file_name = os.getenv("AZURE_FILE_NAME")

file_path = "tendencia_eventos.parquet"
if not os.path.exists(file_path):
    raise FileNotFoundError("Arquivo parquet não encontrado")

In [10]:
print("account_name:", account_name)
print("account_key carregada:", account_key is not None)
print("file_system:", file_system)
print("directory:", directory)
print("file_name:", file_name)

account_name: projetointegradorsg
account_key carregada: True
file_system: bronze
directory: tendencia_eventos
file_name: tendencia_eventos.parquet


In [11]:
print("Arquivo existe:", os.path.exists(file_path))
print("Tamanho (bytes):", os.path.getsize(file_path))

Arquivo existe: True
Tamanho (bytes): 18111


In [12]:
try:
    service_client = DataLakeServiceClient(
        account_url=f"https://{account_name}.dfs.core.windows.net",
        credential=account_key
    )
    
    file_system_client = service_client.get_file_system_client(file_system)
    
    # tentar listar arquivos (teste de conexão)
    paths = list(file_system_client.get_paths())
    
    print("Conexão com Azure OK 🚀")
    print(f"Arquivos encontrados: {len(paths)}")

except Exception as e:
    print("Erro na conexão ❌")
    print(e)

Conexão com Azure OK 🚀
Arquivos encontrados: 0


In [13]:
# Conexão
service_client = DataLakeServiceClient(
    account_url=f"https://{account_name}.dfs.core.windows.net",
    credential=account_key
)

file_system_client = service_client.get_file_system_client(file_system)

# Criar diretório (se não existir)
directory_client = file_system_client.get_directory_client(directory)
if not directory_client.exists():
    directory_client.create_directory()
    print("Diretório criado ✅")
else:
    print("Diretório já existe 👍")

# Upload
file_client = directory_client.get_file_client(file_name)

with open(file_path, "rb") as f:
    file_client.upload_data(f, overwrite=True)

print("Upload concluído 🚀")

Diretório criado ✅
Upload concluído 🚀
